# 7장. RAG 시스템 평가 — 02-1. 한국어 번역 데이터셋으로 RAGAS 평가

## 영어 RAG 시스템, 한국어 질문에도 잘 답할까?

앞 노트북(01, 02)에서는 영어 논문을 영어 질문으로 평가했어요. 하지만 실무에서는 **영어 문서를 한국어로 질문**하는 경우가 자주 있습니다.

이 노트북에서는 RAGAS로 생성한 영어 테스트셋을 **한국어로 번역**한 뒤, 동일한 RAG 시스템을 한국어로 평가해볼게요.

### 학습 목표

- LLM을 활용한 평가 데이터셋 번역 파이프라인 구축
- 한국어 번역 데이터셋으로 RAGAS 평가 실행
- 영어 vs 한국어 평가 결과 비교 및 해석

### 사전 지식

- 01번 노트북에서 합성 테스트셋 생성 완료 (`data/attention_synthetic_testset.csv`)
- 02번 노트북에서 RAGAS 4가지 지표 이해

### 왜 한국어 평가가 필요할까?

```mermaid
flowchart LR
    A[영어 문서<br/>attention_paper.pdf] --> B[RAG 시스템]
    C[한국어 질문] --> B
    B --> D[한국어 답변?]
    D --> E[RAGAS 평가<br/>영어 대비 성능 차이는?]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A,C input
    class B process
    class D,E output
```

영어 임베딩 모델로 구축한 벡터 DB에 한국어 질문이 들어오면 검색 품질이 떨어질 수 있어요. 이 차이를 **정량적으로** 확인하는 것이 이 노트북의 핵심입니다.

---

## 환경 설정

In [1]:
# 필요한 패키지 설치
# !pip install -qU langchain ragas faiss-cpu langchain-openai langchain-community pdfplumber

In [2]:
# DeprecationWarning 억제 (RAGAS 내부 경고)
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# LangSmith 프로젝트 설정
os.environ["LANGSMITH_PROJECT"] = "02_1-Eval-RAGAS-Korean"

---

## 1단계: 영어 테스트셋 로드

01번 노트북에서 생성한 합성 테스트셋을 불러옵니다.

In [3]:
import os

# 데이터 파일 경로 확인
for _path in ["data/attention_synthetic_testset.csv", "data/attention_paper.pdf"]:
    if not os.path.exists(_path):
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {_path}\n01번 노트북을 먼저 실행하세요.")

In [4]:
import pandas as pd
import ast

# 영어 테스트셋 로드
df_en = pd.read_csv("data/attention_synthetic_testset.csv")

# reference_contexts: CSV에서 문자열로 저장된 리스트를 복원
df_en["reference_contexts"] = df_en["reference_contexts"].apply(ast.literal_eval)

print(f"영어 테스트 케이스 수: {len(df_en)}")
print(f"컬럼: {list(df_en.columns)}")

# 첫 번째 질문-답변 확인
print(f"\n[예시]")
print(f"질문: {df_en.iloc[0]['user_input']}")
print(f"정답: {df_en.iloc[0]['reference'][:100]}...")

영어 테스트 케이스 수: 12
컬럼: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']

[예시]
질문: What Transformer do?
정답: The Transformer is a new simple network architecture based solely on attention mechanisms, dispensin...


---

## 2단계: LLM으로 테스트셋 한국어 번역

LLM을 활용해 질문(`user_input`)과 정답(`reference`)을 한국어로 번역합니다.

> **왜 reference_contexts는 번역하지 않나요?** `reference_contexts`는 원본 문서에서 추출한 청크로, RAG 시스템이 검색할 대상과 동일한 언어(영어)여야 합니다. 번역하면 오히려 Context Recall 계산이 부정확해져요.

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 번역용 LLM — 번역 품질을 위해 temperature=0
translate_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 번역 프롬프트
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional English-to-Korean translator specializing in AI/ML technical documents. "
               "Translate the given text naturally into Korean. Keep technical terms in their commonly used Korean forms. "
               "Output ONLY the translated text, nothing else."),
    ("human", "{text}")
])

translate_chain = translate_prompt | translate_llm | StrOutputParser()

# 번역 테스트
test_result = translate_chain.invoke({"text": "What is the Transformer architecture?"})
print(f"번역 테스트: {test_result}")

번역 테스트: 트랜스포머 아키텍처란 무엇인가요?


In [6]:
# ---------------------------------------------------
# 영어 테스트셋의 질문과 정답을 한국어로 번역
# ---------------------------------------------------
# ============================================================
# TODO: batch()로 질문과 정답을 한국어로 번역하세요
# 힌트: translate_chain.batch([{"text": q} for q in questions])
# 예상 결과: 한국어 질문/정답 리스트
# ============================================================

# 질문 번역
questions_en = df_en["user_input"].tolist()
questions_ko = translate_chain.batch([{"text": q} for q in questions_en])

# 정답 번역
references_en = df_en["reference"].tolist()
references_ko = translate_chain.batch([{"text": r} for r in references_en])

print(f"번역 완료: {len(questions_ko)}개의 질문, {len(references_ko)}개의 정답")
print(f"\n[번역 예시]")
for i in range(min(3, len(questions_ko))):
    print(f"\n--- 케이스 {i+1} ---")
    print(f"영어 질문: {questions_en[i]}")
    print(f"한국어 질문: {questions_ko[i]}")
    print(f"한국어 정답: {references_ko[i][:80]}...")

번역 완료: 12개의 질문, 12개의 정답

[번역 예시]

--- 케이스 1 ---
영어 질문: What Transformer do?
한국어 질문: 트랜스포머는 무엇을 하나요?
한국어 정답: 트랜스포머는 주의 메커니즘만을 기반으로 한 새로운 간단한 네트워크 아키텍처로, 순환 및 합성을 완전히 배제합니다. 이는 기계 번역 작업에서 품질...

--- 케이스 2 ---
영어 질문: What contribution did Jakob make to the development of the Transformer model?
한국어 질문: 야곱은 Transformer 모델 개발에 어떤 기여를 했나요?
한국어 정답: 야콥은 RNN을 셀프 어텐션으로 대체하는 것을 제안하고 이 아이디어를 평가하기 위한 노력을 시작했습니다....

--- 케이스 3 ---
영어 질문: How does the Extended Neural GPU contribute to the efficiency of machine learning models compared to traditional recurrent networks?
한국어 질문: 확장 신경 GPU(Extended Neural GPU)는 전통적인 순환 신경망에 비해 머신러닝 모델의 효율성에 어떻게 기여하나요?
한국어 정답: 확장된 신경 GPU는 ByteNet 및 ConvS2S와 같은 모델과 함께 기본 구성 요소로 합성곱 신경망을 사용하여 모든 입력 및 출력 위치에 ...


In [7]:
# ---------------------------------------------------
# 한국어 번역 데이터셋 생성 및 저장
# ---------------------------------------------------
df_ko = pd.DataFrame({
    "user_input": questions_ko,
    "reference": references_ko,
    # reference_contexts는 원본 영어 그대로 유지
    "reference_contexts": df_en["reference_contexts"].apply(str).tolist(),
    "synthesizer_name": df_en["synthesizer_name"].tolist(),
})

# CSV 저장
output_path = "data/attention_synthetic_testset_korean.csv"
df_ko.to_csv(output_path, index=False)

print(f"한국어 테스트셋 저장: {output_path}")
print(f"총 {len(df_ko)}개의 테스트 케이스")
df_ko.head()

한국어 테스트셋 저장: data/attention_synthetic_testset_korean.csv
총 12개의 테스트 케이스


,user_input,reference,reference_contexts,synthesizer_name
0,트랜스포머는 무엇을 하나요?,"트랜스포머는 주의 메커니즘만을 기반으로 한 새로운 간단한 네트워크 아키텍처로, 순환...","['Providedproperattributionisprovided,Googlehe...",single_hop_specific_query_synthesizer
1,야곱은 Transformer 모델 개발에 어떤 기여를 했나요?,야콥은 RNN을 셀프 어텐션으로 대체하는 것을 제안하고 이 아이디어를 평가하기 위한...,"['trainingfor3.5daysoneightGPUs,asmallfraction...",single_hop_specific_query_synthesizer
2,확장 신경 GPU(Extended Neural GPU)는 전통적인 순환 신경망에 비...,확장된 신경 GPU는 ByteNet 및 ConvS2S와 같은 모델과 함께 기본 구성...,['areusedinconjunctionwitharecurrentnetwork.\n...,single_hop_specific_query_synthesizer
3,머신러닝에서 점곱 주의(attention) 메커니즘은 입력 데이터의 특정 부분에 집...,점곱 주의(attention) 메커니즘은 쿼리와 해당 키의 호환성 함수를 계산하는 ...,['ScaledDot-ProductAttention Multi-HeadAttenti...,single_hop_specific_query_synthesizer
4,"저자들은 계산 복잡성과 다양한 레이어 유형의 최대 경로 길이를 고려할 때, 학습된 ...",저자들은 자기 주의 레이어에 대해 사인파 형태의 위치 인코딩을 선택한 이유는 모델이...,"['<1-hop>\n\nrelativepositions,sinceforanyfixe...",multi_hop_abstract_query_synthesizer


---

## 3단계: RAG 시스템 구축

02번 노트북과 동일한 RAG 시스템을 구축합니다. Attention 논문 PDF를 기반으로 합니다.

In [8]:
# ---------------------------------------------------
# RAG 시스템 구축 (문서 로드 → 분할 → 벡터화 → 체인 조립)
# ---------------------------------------------------
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 1단계: 문서 로드
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()
print(f"문서 로드 완료: {len(docs)} 페이지")

# 2단계: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"문서 분할 완료: {len(split_documents)} 청크")

# 3단계: 임베딩 및 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)
print("벡터 DB 생성 완료")

# 4단계: Retriever 생성
retriever = vectorstore.as_retriever()

# 5단계: 프롬프트 — 한국어 질문에도 대응하도록 설계
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks about the Transformer architecture.
Use the following pieces of retrieved context to answer the question.
If the question is in Korean, answer in Korean.
If you don't know the answer, just say that you don't know.

#Context:
{context}

#Question:
{question}

#Answer:"""
)

# 6단계: LLM 및 RAG 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 시스템 구축 완료")

문서 로드 완료: 15 페이지
문서 분할 완료: 48 청크
벡터 DB 생성 완료
RAG 시스템 구축 완료


### RAG 시스템 동작 확인 — 영어 vs 한국어

In [9]:
# 영어 질문
en_answer = chain.invoke("What is multi-head attention?")
print("[영어 질문]")
print(f"답변: {en_answer[:200]}...")

print()

# 한국어 질문
ko_answer = chain.invoke("멀티헤드 어텐션이란 무엇인가요?")
print("[한국어 질문]")
print(f"답변: {ko_answer[:200]}...")

[영어 질문]
답변: 멀티-헤드 어텐션(Multi-Head Attention)은 트랜스포머 아키텍처에서 사용되는 중요한 구성 요소로, 여러 개의 어텐션 레이어를 병렬로 실행하여 입력 데이터의 다양한 표현 하위 공간에서 정보를 동시에 주목할 수 있게 합니다. 각 어텐션 헤드는 서로 다른 선형 변환을 통해 쿼리, 키, 값의 차원을 줄인 후, 이들에 대해 어텐션 함수를 수행합니다. ...

[한국어 질문]
답변: 멀티헤드 어텐션은 트랜스포머 아키텍처에서 사용되는 어텐션 메커니즘의 한 형태로, 여러 개의 어텐션 헤드를 동시에 사용하는 방식입니다. 각 헤드는 입력의 서로 다른 부분에 집중하여 다양한 표현을 학습할 수 있도록 합니다. 이렇게 함으로써 모델은 입력 데이터의 다양한 측면을 동시에 고려할 수 있으며, 더 풍부한 정보를 추출할 수 있습니다. 멀티헤드 어텐션은 각...


---

## 4단계: 한국어 테스트셋으로 답변 생성

한국어로 번역한 질문을 RAG 시스템에 입력하고 답변을 생성합니다.

In [10]:
from datasets import Dataset

# 한국어 테스트셋을 Dataset으로 변환
df_ko_loaded = pd.read_csv("data/attention_synthetic_testset_korean.csv")
df_ko_loaded["reference_contexts"] = df_ko_loaded["reference_contexts"].apply(ast.literal_eval)

test_dataset_ko = Dataset.from_pandas(df_ko_loaded[["user_input", "reference", "reference_contexts"]])

# 한국어 질문 리스트
batch_questions_ko = [q for q in test_dataset_ko["user_input"]]

print(f"총 {len(batch_questions_ko)}개의 한국어 질문에 대해 답변 생성")
print(f"\n질문 예시:")
for i, q in enumerate(batch_questions_ko[:3], 1):
    print(f"  {i}. {q}")

총 12개의 한국어 질문에 대해 답변 생성

질문 예시:
  1. 트랜스포머는 무엇을 하나요?
  2. 야곱은 Transformer 모델 개발에 어떤 기여를 했나요?
  3. 확장 신경 GPU(Extended Neural GPU)는 전통적인 순환 신경망에 비해 머신러닝 모델의 효율성에 어떻게 기여하나요?


In [11]:
# ---------------------------------------------------
# 한국어 질문에 대한 RAG 답변 배치 생성
# ---------------------------------------------------
# ============================================================
# TODO: chain.batch()로 한국어 질문에 답변을 생성하세요
# 힌트: answers_ko = chain.batch(batch_questions_ko)
# 예상 결과: len(answers_ko) == len(batch_questions_ko)
# ============================================================

answers_ko = chain.batch(batch_questions_ko)

print(f"답변 생성 완료: {len(answers_ko)}개")
print(f"\n답변 예시:")
for i, ans in enumerate(answers_ko[:2], 1):
    print(f"\n{i}. {ans[:200]}...")

답변 생성 완료: 12개

답변 예시:

1. 트랜스포머는 주로 기계 번역과 같은 자연어 처리 작업을 수행하는 데 사용되는 모델입니다. 이 모델은 입력 시퀀스의 각 요소 간의 관계를 학습하기 위해 자기 주의 메커니즘(self-attention mechanism)을 사용하며, 이를 통해 문맥을 이해하고 번역 품질을 향상시킵니다. 트랜스포머는 또한 병렬 처리에 유리하여 훈련 시간을 크게 단축시킬 수 있습니...

2. 야곱은 RNN을 자기 주의(attention) 메커니즘으로 대체하는 아이디어를 제안하였고, 이 아이디어를 평가하는 노력을 시작했습니다....


In [12]:
# 답변을 데이터셋에 추가
if "answer" in test_dataset_ko.column_names:
    test_dataset_ko = test_dataset_ko.remove_columns(["answer"])

test_dataset_ko = test_dataset_ko.add_column("answer", answers_ko)

# ---------------------------------------------------
# retrieved_contexts 추가: RAG 검색기가 실제로 찾은 컨텍스트
# ---------------------------------------------------
# RAGAS 0.4.x는 두 종류의 컨텍스트를 구분:
#   - reference_contexts: 테스트셋 생성 시 정답 근거로 사용된 컨텍스트 (정답)
#   - retrieved_contexts: RAG 검색기가 실제로 찾아온 컨텍스트 (평가 대상)

print("검색기로 각 한국어 질문의 컨텍스트 검색 중...")
retrieved_contexts_ko = []
for q in batch_questions_ko:
    docs_found = retriever.invoke(q)
    retrieved_contexts_ko.append([doc.page_content for doc in docs_found])

if "retrieved_contexts" in test_dataset_ko.column_names:
    test_dataset_ko = test_dataset_ko.remove_columns(["retrieved_contexts"])
test_dataset_ko = test_dataset_ko.add_column("retrieved_contexts", retrieved_contexts_ko)

print(f"Dataset 컬럼: {test_dataset_ko.column_names}")
print(f"\n[첫 번째 케이스]")
print(f"질문: {test_dataset_ko[0]['user_input']}")
print(f"답변: {test_dataset_ko[0]['answer'][:150]}...")
print(f"정답: {test_dataset_ko[0]['reference'][:150]}...")
print(f"검색된 컨텍스트 수: {len(test_dataset_ko[0]['retrieved_contexts'])}")

검색기로 각 한국어 질문의 컨텍스트 검색 중...
Dataset 컬럼: ['user_input', 'reference', 'reference_contexts', 'answer', 'retrieved_contexts']

[첫 번째 케이스]
질문: 트랜스포머는 무엇을 하나요?
답변: 트랜스포머는 주로 기계 번역과 같은 자연어 처리 작업을 수행하는 데 사용되는 모델입니다. 이 모델은 입력 시퀀스의 각 요소 간의 관계를 학습하기 위해 자기 주의 메커니즘(self-attention mechanism)을 사용하며, 이를 통해 문맥을 이해하고 번역 품질을 ...
정답: 트랜스포머는 주의 메커니즘만을 기반으로 한 새로운 간단한 네트워크 아키텍처로, 순환 및 합성을 완전히 배제합니다. 이는 기계 번역 작업에서 품질 면에서 우수함을 보여주었으며, 병렬화가 더 용이하고 훈련에 필요한 시간이 현저히 적습니다....
검색된 컨텍스트 수: 4


---

## 5단계: RAGAS 평가 실행 (한국어)

한국어 데이터셋으로 RAGAS 4가지 지표를 평가합니다.

> **주의**: `reference_contexts`는 영어 원본 그대로이고, `answer`는 한국어입니다. Faithfulness 지표는 답변의 각 주장(claim)이 컨텍스트에서 확인 가능한지 검사하는데, 언어가 다르면 LLM이 교차 언어 추론을 해야 하므로 점수가 달라질 수 있어요.

In [13]:
# ---------------------------------------------------
# RAGAS 4가지 지표 평가 실행 (한국어 데이터셋)
# ---------------------------------------------------
# ============================================================
# TODO: evaluate()로 한국어 데이터셋을 평가하세요
# 힌트: 02번 노트북과 동일한 방식, dataset만 한국어 데이터셋으로 변경
# 예상 결과: 4가지 메트릭 점수가 담긴 result 객체
# ============================================================

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# RAGAS 평가용 LLM과 Embeddings
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

print("한국어 RAGAS 평가 시작...")
print("=" * 80)

result_ko = evaluate(
    dataset=test_dataset_ko,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n한국어 평가 완료!")
print("=" * 80)
result_ko

한국어 RAGAS 평가 시작...


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



한국어 평가 완료!


{'context_precision': 0.6991, 'context_recall': 0.6319, 'faithfulness': 0.6283, 'answer_relevancy': 0.3605}

---

## 6단계: 영어 평가 실행 (비교용)

동일한 RAG 시스템에 영어 테스트셋으로도 평가를 실행해서 비교합니다.

In [14]:
# ---------------------------------------------------
# 영어 테스트셋으로 답변 생성 및 평가 (비교용)
# ---------------------------------------------------

# 영어 Dataset 준비
test_dataset_en = Dataset.from_pandas(df_en[["user_input", "reference", "reference_contexts"]])

# 영어 질문으로 답변 생성
batch_questions_en = [q for q in test_dataset_en["user_input"]]
answers_en = chain.batch(batch_questions_en)

# 답변 추가
if "answer" in test_dataset_en.column_names:
    test_dataset_en = test_dataset_en.remove_columns(["answer"])
test_dataset_en = test_dataset_en.add_column("answer", answers_en)

# retrieved_contexts 추가
print("검색기로 각 영어 질문의 컨텍스트 검색 중...")
retrieved_contexts_en = []
for q in batch_questions_en:
    docs_found = retriever.invoke(q)
    retrieved_contexts_en.append([doc.page_content for doc in docs_found])

if "retrieved_contexts" in test_dataset_en.column_names:
    test_dataset_en = test_dataset_en.remove_columns(["retrieved_contexts"])
test_dataset_en = test_dataset_en.add_column("retrieved_contexts", retrieved_contexts_en)

print(f"영어 답변 생성 완료: {len(answers_en)}개")
print(f"Dataset 컬럼: {test_dataset_en.column_names}")

검색기로 각 영어 질문의 컨텍스트 검색 중...
영어 답변 생성 완료: 12개
Dataset 컬럼: ['user_input', 'reference', 'reference_contexts', 'answer', 'retrieved_contexts']


In [15]:
print("영어 RAGAS 평가 시작...")
print("=" * 80)

result_en = evaluate(
    dataset=test_dataset_en,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n영어 평가 완료!")
print("=" * 80)
result_en

영어 RAGAS 평가 시작...


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



영어 평가 완료!


{'context_precision': 0.8171, 'context_recall': 0.9000, 'faithfulness': 0.9223, 'answer_relevancy': 0.7628}

---

## 7단계: 영어 vs 한국어 비교 분석

In [16]:
# ---------------------------------------------------
# 영어 vs 한국어 RAGAS 점수 비교
# ---------------------------------------------------
metric_names = ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]

result_en_df = result_en.to_pandas()
result_ko_df = result_ko.to_pandas()

print("=" * 80)
print("영어 vs 한국어 RAGAS 평가 결과 비교")
print("=" * 80)
print(f"{'메트릭':<25} {'영어':>8} {'한국어':>8} {'차이':>8}")
print("-" * 55)

for metric in metric_names:
    en_score = result_en_df[metric].mean()
    ko_score = result_ko_df[metric].mean()
    diff = ko_score - en_score
    arrow = "↑" if diff > 0 else "↓" if diff < 0 else "="
    print(f"{metric:<25} {en_score:>8.4f} {ko_score:>8.4f} {diff:>+7.4f} {arrow}")

print("-" * 55)
en_avg = result_en_df[metric_names].mean().mean()
ko_avg = result_ko_df[metric_names].mean().mean()
print(f"{'전체 평균':<25} {en_avg:>8.4f} {ko_avg:>8.4f} {ko_avg - en_avg:>+7.4f}")

영어 vs 한국어 RAGAS 평가 결과 비교
메트릭                             영어      한국어       차이
-------------------------------------------------------
context_precision           0.8171   0.6991 -0.1181 ↓
context_recall              0.9000   0.6319 -0.2681 ↓
faithfulness                0.9223   0.6283 -0.2940 ↓
answer_relevancy            0.7628   0.3605 -0.4023 ↓
-------------------------------------------------------
전체 평균                       0.8505   0.5800 -0.2706


In [17]:
# ---------------------------------------------------
# 케이스별 상세 비교
# ---------------------------------------------------
print("=" * 80)
print("케이스별 Faithfulness 비교 (할루시네이션 체크)")
print("=" * 80)

for i in range(len(result_en_df)):
    en_faith = result_en_df.iloc[i]["faithfulness"]
    ko_faith = result_ko_df.iloc[i]["faithfulness"]
    diff = ko_faith - en_faith
    
    # 점수 차이가 큰 케이스 강조
    marker = " ◀ 주목" if abs(diff) > 0.3 else ""
    print(f"  [{i+1:2d}] 영어: {en_faith:.2f} | 한국어: {ko_faith:.2f} | 차이: {diff:+.2f}{marker}")
    if abs(diff) > 0.3:
        print(f"       질문(KO): {result_ko_df.iloc[i]['user_input'][:60]}...")

케이스별 Faithfulness 비교 (할루시네이션 체크)
  [ 1] 영어: 1.00 | 한국어: 0.80 | 차이: -0.20
  [ 2] 영어: 1.00 | 한국어: 1.00 | 차이: +0.00
  [ 3] 영어: 0.89 | 한국어: 0.56 | 차이: -0.33 ◀ 주목
       질문(KO): 확장 신경 GPU(Extended Neural GPU)는 전통적인 순환 신경망에 비해 머신러닝 모델의 효율성...
  [ 4] 영어: 0.90 | 한국어: 0.86 | 차이: -0.04
  [ 5] 영어: 0.83 | 한국어: 1.00 | 차이: +0.17
  [ 6] 영어: 1.00 | 한국어: 0.67 | 차이: -0.33 ◀ 주목
       질문(KO): 트랜스포머 모델 아키텍처에서 인코더는 입력 데이터를 처리하기 위해 자기 주의(self-attention)와 ...
  [ 7] 영어: 0.83 | 한국어: 0.88 | 차이: +0.04
  [ 8] 영어: 0.89 | 한국어: 0.29 | 차이: -0.60 ◀ 주목
       질문(KO): Scaled Dot-Product Attention과 Additive Attention의 성능 차이는 주로 ...
  [ 9] 영어: 1.00 | 한국어: 0.00 | 차이: -1.00 ◀ 주목
       질문(KO): 멀티 헤드 어텐션은 스케일드 닷 프로덕트 어텐션과 같은 전통적인 어텐션 메커니즘에 비해 트랜스포머 모델의 성...
  [10] 영어: 1.00 | 한국어: 0.00 | 차이: -1.00 ◀ 주목
       질문(KO): 멀티 헤드 어텐션은 스케일드 닷 프로덕트 어텐션과 같은 전통적인 어텐션 메커니즘에 비해 트랜스포머 모델의 성...
  [11] 영어: 0.92 | 한국어: 1.00 | 차이: +0.08
  [12] 영어: 0.80 | 한국어: 0.50 | 차이: -0.30 ◀ 주목
       질문(KO): 구글 리서치에서 수행된 작업이 트랜스포머 모델의 개발에 어떻게 기여했으며, 이것이 기

### 결과 해석 가이드

| 패턴 | 의미 | 원인 |
|------|------|------|
| Context Precision/Recall 하락 | 한국어 질문으로 영어 문서 검색 품질 저하 | 임베딩 모델의 교차 언어 성능 한계 |
| Faithfulness 하락 | 한국어 답변에서 할루시네이션 증가 | 영어→한국어 변환 시 LLM이 원문에 없는 내용 추가 |
| Answer Relevancy 유사 | 한국어로도 질문에 맞는 답변 생성 | LLM의 다국어 능력이 양호 |
| 전체적으로 유사 | 모델이 다국어에 강함 | text-embedding-3-small이 한국어도 잘 처리 |

---

## 평가 결과 저장

In [18]:
# 한국어 평가 결과 저장
ko_output_path = "data/attention_evaluation_result_korean.csv"
result_ko_df.to_csv(ko_output_path, index=False)

print(f"한국어 평가 결과 저장: {ko_output_path}")
print(f"총 {len(result_ko_df)}개의 평가 케이스")

한국어 평가 결과 저장: data/attention_evaluation_result_korean.csv
총 12개의 평가 케이스


---

## 정리

### 한국어 번역 평가 워크플로우

| 단계 | 작업 | 핵심 포인트 |
|------|------|:-----------:|
| 1. 데이터 준비 | 영어 합성 테스트셋 로드 | `attention_synthetic_testset.csv` |
| 2. 번역 | LLM으로 질문/정답 한국어 번역 | `reference_contexts`는 영어 유지 |
| 3. RAG 구축 | 영어 문서 기반 RAG 시스템 | 한국어 질문 대응 프롬프트 |
| 4. 답변 생성 | 한국어 질문으로 RAG 답변 | `chain.batch()` |
| 5. RAGAS 평가 | 4가지 지표 평가 | 영어와 동일한 지표 |
| 6. 비교 분석 | 영어 vs 한국어 점수 비교 | 교차 언어 성능 차이 정량화 |

### 교차 언어 RAG 평가에서 얻은 인사이트

1. **임베딩 모델 선택이 핵심**: 영어 전용 임베딩 모델은 한국어 검색에 취약할 수 있음
2. **프롬프트 설계**: "한국어로 답변" 지시를 추가하면 Answer Relevancy가 개선됨
3. **Faithfulness 주의**: 언어 변환 과정에서 할루시네이션 발생 가능성이 높아질 수 있음
4. **다국어 임베딩 고려**: 다국어 서비스라면 `multilingual-e5-large` 같은 모델을 검토

### 개선 방향

| 문제 | 해결 방법 |
|------|----------|
| Context Precision 하락 | 다국어 임베딩 모델 사용 (e.g., `multilingual-e5-large`) |
| Context Recall 하락 | 검색 문서 수(k) 증가, 쿼리 번역(한→영) 후 검색 |
| Faithfulness 하락 | 프롬프트에 "컨텍스트 기반 답변" 강화 |
| Answer Relevancy 하락 | 프롬프트에 언어 지시 명확화 |